# Monte Carlo convergence

Checks on `pricing.monte_carlo.mc_price` against the closed-form Black-Scholes price:

1. **Absolute error vs. N** (log-log): should decay with slope approximately $-1/2$.
2. **Standard error vs. N** (log-log): same expected slope, since $\text{SE} = \sigma_X/\sqrt{N}$.
3. **Plain vs. antithetic**: antithetic variates should not change the $-1/2$ slope, but should sit below the plain estimator (lower variance for the same N).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pricing.black_scholes import black_scholes_price
from pricing.monte_carlo import mc_price

In [ ]:
# Market parameters (same as the Black-Scholes notebook, for comparability)
S0 = 100.0
K = 100.0
T = 1.0
r = 0.05
sigma = 0.20
q = 0.02

bs_call_price = black_scholes_price(S0, K, T, r, sigma, option_type="call", q=q)
print(f"Black-Scholes reference price: {bs_call_price:.6f}")

## Convergence study: plain vs. antithetic Monte Carlo

In [ ]:
N_values = np.unique(np.logspace(2, 6, 13).astype(int))

rows = []
for N in N_values:
    res_plain = mc_price(S0, K, T, r, sigma, option_type="call", q=q, n_paths=N, antithetic=False, seed=42)
    res_anti = mc_price(S0, K, T, r, sigma, option_type="call", q=q, n_paths=N, antithetic=True, seed=42)
    rows.append({
        "N": N,
        "price_plain": res_plain.price,
        "std_error_plain": res_plain.std_error,
        "abs_error_plain": abs(res_plain.price - bs_call_price),
        "price_antithetic": res_anti.price,
        "std_error_antithetic": res_anti.std_error,
        "abs_error_antithetic": abs(res_anti.price - bs_call_price),
    })

df = pd.DataFrame(rows)
df

## Absolute error vs. N (log-log, slope -1/2 reference)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(df["N"], df["abs_error_plain"], "o-", label="Absolute error (plain)")
ax.loglog(df["N"], df["abs_error_antithetic"], "o-", label="Absolute error (antithetic)")

# Reference line with slope -1/2, anchored to the first plain-MC point
C = df["abs_error_plain"].iloc[0] * np.sqrt(df["N"].iloc[0])
ax.loglog(df["N"], C / np.sqrt(df["N"]), "k--", label="Slope -1/2 reference")

ax.set_xlabel("Number of paths N")
ax.set_ylabel("Absolute error vs Black-Scholes")
ax.set_title("Monte Carlo convergence: absolute error vs N")
ax.legend()
fig.tight_layout()
plt.show()

## Standard error vs. N (log-log, slope -1/2 reference)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(df["N"], df["std_error_plain"], "o-", label="Std. error (plain)")
ax.loglog(df["N"], df["std_error_antithetic"], "o-", label="Std. error (antithetic)")

C2 = df["std_error_plain"].iloc[0] * np.sqrt(df["N"].iloc[0])
ax.loglog(df["N"], C2 / np.sqrt(df["N"]), "k--", label="Slope -1/2 reference")

ax.set_xlabel("Number of paths N")
ax.set_ylabel("Monte Carlo standard error")
ax.set_title("Monte Carlo convergence: standard error vs N")
ax.legend()
fig.tight_layout()
plt.show()

## Comparison table: plain vs. antithetic

In [ ]:
comparison = df[["N", "price_plain", "std_error_plain", "price_antithetic", "std_error_antithetic"]].copy()
comparison["variance_reduction_factor"] = (
    comparison["std_error_plain"] / comparison["std_error_antithetic"]
)
comparison

## Appendix: what the simulated paths actually look like

`mc_price` never builds a full time path: for a European option only $S_T$ matters, so it samples the terminal value directly from its closed-form lognormal distribution. This section is purely illustrative — it shows the kind of random trajectories that $S_T$ is secretly drawn from, by stepping through time instead of jumping straight to the end.

Path-dependent payoffs (Asian, barrier, lookback — see `exotics.py` in a later stage) *do* need this full path, not just its endpoint.

In [ ]:
n_display_paths = 30
n_steps = 252  # trading days
dt = T / n_steps

rng_paths = np.random.default_rng(7)  # separate seed: illustration only, not used for pricing
z = rng_paths.standard_normal((n_display_paths, n_steps))
log_increments = (r - q - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z
log_paths = np.cumsum(log_increments, axis=1)
paths = S0 * np.exp(np.hstack([np.zeros((n_display_paths, 1)), log_paths]))

time_grid = np.linspace(0, T, n_steps + 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(time_grid, paths.T, linewidth=0.8, alpha=0.7)
ax.axhline(K, color="black", linestyle="--", linewidth=1, label="Strike K")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Simulated price $S_t$")
ax.set_title(f"{n_display_paths} simulated GBM paths under the risk-neutral measure")
ax.legend()
fig.tight_layout()
plt.show()